# Task 2 — Technical indicators (TA-Lib + PyNance)

Download daily **OHLCV** prices, clean types and missing values, compute **SMA / EMA**, **RSI**, **MACD** with **TA-Lib**, add **PyNance** session return and rolling volatility, then plot price with moving averages and separate indicator panels.

**Data:** If `data/raw/stock_prices.csv` exists (Date + Open/High/Low/Close/Volume), it is loaded first. Otherwise prices are downloaded with **`yfinance`** using `TICKER` and `PRICE_PERIOD`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import talib
import yfinance as yf
import pynance.tech as pnt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.stock_prices import load_ohlcv_csv, ohlcv_missing_report, prepare_price_df

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 3)


## Load prices (`yfinance` or optional local CSV)


In [ ]:
TICKER = "AAPL"
PRICE_PERIOD = "5y"  # yfinance only: e.g. "1y", "5y", "max"
PLOT_LAST_N = 600  # most recent trading days in charts (None = full series)
LOCAL_OHLCV = ROOT / "data" / "raw" / "stock_prices.csv"

if LOCAL_OHLCV.is_file():
    print("Loading local CSV:", LOCAL_OHLCV)
    raw = load_ohlcv_csv(LOCAL_OHLCV)
else:
    print("Downloading Yahoo:", TICKER, PRICE_PERIOD)
    raw = yf.Ticker(TICKER).history(period=PRICE_PERIOD, auto_adjust=True, actions=True)

df = prepare_price_df(raw)
drop_misc = [
    c
    for c in df.columns
    if c not in ("Open", "High", "Low", "Close", "Volume", "Adj Close")
]
df = df.drop(columns=drop_misc, errors="ignore")

print("Rows:", len(df))
print("Missing values (before fill):")
print(ohlcv_missing_report(df))

ohlcv = ["Open", "High", "Low", "Close", "Volume"]
df[ohlcv] = df[ohlcv].ffill()
df = df.dropna(subset=["Close", "Volume"])

print("Missing values (after ffill + drop):")
print(ohlcv_missing_report(df))
df.tail()


## TA-Lib: SMA, EMA, RSI, MACD


In [ ]:
close = df["Close"].values.astype(float)

df["SMA_20"] = talib.SMA(close, timeperiod=20)
df["SMA_50"] = talib.SMA(close, timeperiod=50)
df["EMA_12"] = talib.EMA(close, timeperiod=12)
df["EMA_26"] = talib.EMA(close, timeperiod=26)
df["RSI_14"] = talib.RSI(close, timeperiod=14)
macd, macds, macdh = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
df["MACD"] = macd
df["MACD_signal"] = macds
df["MACD_hist"] = macdh
df[["Close", "SMA_20", "SMA_50", "RSI_14"]].tail()


## PyNance: session return and rolling volatility (`Adj Close`)


In [ ]:
pn_ret = pnt.ret(df[["Adj Close"]], n_sessions=1, outputcol="PnReturn1d")
pn_vol = pnt.volatility(df[["Adj Close"]], window=20, outputcol="PnVol20")
df = df.join(pn_ret, how="left").join(pn_vol, how="left")
df[["Close", "PnReturn1d", "PnVol20"]].tail()


## Visualizations


In [ ]:
plot_df = df if PLOT_LAST_N is None else df.iloc[-PLOT_LAST_N:]

fig, axes = plt.subplots(
    5,
    1,
    figsize=(12, 12),
    sharex=True,
    gridspec_kw={"height_ratios": [2.4, 1.0, 1.0, 0.75, 0.75]},
)

ax0 = axes[0]
ax0.plot(plot_df.index, plot_df["Close"], label="Close", color="black", linewidth=1.0)
ax0.plot(plot_df.index, plot_df["SMA_20"], label="SMA 20", alpha=0.85)
ax0.plot(plot_df.index, plot_df["SMA_50"], label="SMA 50", alpha=0.85)
ax0.plot(plot_df.index, plot_df["EMA_12"], label="EMA 12", alpha=0.85)
ax0.plot(plot_df.index, plot_df["EMA_26"], label="EMA 26", alpha=0.85)
ax0.set_title(f"{TICKER} — close with moving averages")
ax0.set_ylabel("Price")
ax0.legend(loc="upper left", ncol=3, fontsize=8)

ax1 = axes[1]
ax1.plot(plot_df.index, plot_df["RSI_14"], color="purple", linewidth=1)
ax1.axhline(70, color="red", linestyle="--", linewidth=0.8, label="Overbought (70)")
ax1.axhline(30, color="green", linestyle="--", linewidth=0.8, label="Oversold (30)")
ax1.set_ylabel("RSI(14)")
ax1.set_ylim(0, 100)
ax1.legend(loc="upper right", fontsize=7)

ax2 = axes[2]
ax2.plot(plot_df.index, plot_df["MACD"], label="MACD", color="blue")
ax2.plot(plot_df.index, plot_df["MACD_signal"], label="Signal", color="orange")
ax2.bar(plot_df.index, plot_df["MACD_hist"], label="Hist", color="gray", alpha=0.45, width=1.0)
ax2.set_ylabel("MACD")
ax2.legend(loc="upper left", ncol=3, fontsize=7)

ax3 = axes[3]
ax3.plot(plot_df.index, plot_df["PnReturn1d"] * 100, color="darkred", linewidth=0.9, label="Pn return 1d (%)")
ax3.axhline(0, color="black", linewidth=0.5, linestyle=":")
ax3.set_ylabel("Pn ret %")
ax3.legend(loc="upper left", fontsize=7)

ax4 = axes[4]
ax4.plot(plot_df.index, plot_df["PnVol20"], color="teal", label="PyNance vol 20d")
ax4.set_ylabel("Pn vol")
ax4.set_xlabel("Date")
ax4.legend(loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()


## Data preparation summary (for your report)

- **Source:** Prefer `data/raw/stock_prices.csv` when present (daily OHLCV + Date). Else **`yfinance`** with `auto_adjust=True` so **Close** is split/dividend-adjusted; **`Adj Close`** is set to **Close** when Yahoo omits a separate column for PyNance.
- **Types:** OHLCV coerced to float64; index sorted; duplicate timestamps removed (`prepare_price_df`).
- **Missing values:** `ohlcv_missing_report` counts; **`ffill`** on OHLCV then drop rows missing **Close**/**Volume**. Leading **NaN**s on long windows are expected until indicators warm up.
- **Indicators:** **TA-Lib** — SMA(20,50), EMA(12,26), RSI(14), MACD(12,26,9). **PyNance** — 1-session return, 20-day rolling volatility on adjusted close.
